# Adaptive ELPD-Blend Attack — Full Experiment

**Goal:** Produce the Query vs ASR curve proving the ELPD-Blend attack reaches >90% ASR
significantly earlier than MI-FGSM, DI-FGSM, NES-only, Static-Hybrid, and Square Attack.

**Threat model:** L∞, ε=0.05, target=ResNet-50, surrogate=VGG-16-BN, dataset=ImageNet.

Run cells in order.  The experiment auto-saves a CSV and (optionally) logs to W&B.

In [1]:
# ── Cell 1: Environment setup ─────────────────────────────────────────────
import os, sys, logging, time
import torch
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Subset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(name)s — %(message)s',
)

PROJECT_ROOT = os.path.abspath('.')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.utils.config_loader import load_config, dump_config
from src.utils.logger        import RunLogger
from src.models.model_loader import TargetModel, SurrogateModel
from src.attacks.elpd_blender   import ELPDBlender
from src.attacks.query_estimator import QueryEstimator
from src.attacks.elpd_attack    import elpd_blend_attack
from src.attacks.baselines      import (
    run_mifgsm, run_difgsm, run_nes_only,
    run_static_hybrid, run_square_attack,
)

cfg    = load_config('configs/config.yaml')
DEVICE = torch.device(cfg.experiment.device if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
torch.manual_seed(cfg.experiment.seed)

2026-05-10 17:36:54,472 INFO src.utils.config_loader — Loaded config from /Users/eyoel/Documents/projects/adversarial_ml/configs/config.yaml


Device: cpu


In [2]:
# ── Cell 2: Save config snapshot for reproducibility ──────────────────────
os.makedirs('results/logs', exist_ok=True)
dump_config(cfg, f'results/logs/config_snapshot_{cfg.experiment.name}.yaml')
print('Config snapshot saved.')

2026-05-10 17:37:00,391 INFO src.utils.config_loader — Config snapshot saved to results/logs/config_snapshot_elpd_blend_attack_v1.yaml


Config snapshot saved.


In [3]:
# ── Cell 3: Dataset ───────────────────────────────────────────────────────
# AutoDL: place ImageNet validation set at the path in config.yaml (dataset.root).
# The validation folder should follow torchvision ImageFolder format:
#   {root}/val/{class_name}/{image}.JPEG
#
# For a quick sanity check with fewer images, set N_IMAGES < cfg.dataset.num_samples.
N_IMAGES = cfg.dataset.num_samples   # set to e.g. 50 for a fast dry run

transform = T.Compose([
    T.Resize(256),
    T.CenterCrop(cfg.dataset.image_size),
    T.ToTensor(),         # → [0,1]
    # Note: we do NOT normalise here — normalisation happens inside the model wrappers
    # so that the attack operates in the natural [0,1] pixel space.
])

val_dataset = torchvision.datasets.ImageFolder(
    root=os.path.join(cfg.dataset.root, 'val'),
    transform=transform,
)

# Deterministically sample N_IMAGES images (balanced across classes)
torch.manual_seed(cfg.experiment.seed)
indices = torch.randperm(len(val_dataset))[:N_IMAGES].tolist()
subset  = Subset(val_dataset, indices)
loader  = DataLoader(subset, batch_size=1, shuffle=False, num_workers=cfg.dataset.num_workers)

print(f'Dataset: {cfg.dataset.name}, {N_IMAGES} images loaded.')

FileNotFoundError: [Errno 2] No such file or directory: '/Users/eyoel/Documents/projects/adversarial_ml/data/val'

In [4]:
# ── Cell 4: Load models ───────────────────────────────────────────────────
# Both models download pretrained weights from torchvision on first run.
# AutoDL pods have internet access; weights are cached at ~/.cache/torch.

surrogate = SurrogateModel(cfg.models.surrogate, cfg.dataset, device=DEVICE)
print(f'Surrogate loaded: {cfg.models.surrogate.arch}')

# TargetModel is instantiated per-image (label changes); we create the shell here
# and call .set_label() in the loop below.
print(f'Target arch: {cfg.models.target.arch} (loaded per-image in loop)')

/Users/eyoel/Documents/projects/adversarial_ml/.venv/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/eyoel/Documents/projects/adversarial_ml/.venv/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_BN_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_BN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Surrogate loaded: vgg16_bn
Target arch: resnet50 (loaded per-image in loop)


In [6]:
# ── Cell 5: Main evaluation loop ─────────────────────────────────────────
# Results accumulator: one row per image per method
records = []

# Pre-build one TargetModel shell — we'll reuse it across images
target_model = TargetModel(
    cfg.models.target, cfg.dataset,
    query_budget=cfg.attack.query_budget,
    device=DEVICE,
    label=0,
)

run_logger = RunLogger(cfg)

for img_idx, (x_batch, y_batch) in enumerate(loader):
    x_orig = x_batch[0].to(DEVICE)       # (C, H, W)
    label  = int(y_batch[0].item())

    # Skip images the target already misclassifies (no attack needed)
    target_model.set_label(label)
    target_model.reset_count()
    if target_model.predict(x_orig) != label:
        print(f'[{img_idx:04d}] Skip — target already wrong on clean image.')
        continue

    print(f'[{img_idx:04d}] label={label}  ', end='', flush=True)
    run_logger.start_image(img_idx, label)

    # ── ELPD-Blend attack ─────────────────────────────────────────────────
    target_model.reset_count()
    blender   = ELPDBlender(cfg.elpd_blender, sigma=cfg.query_estimator.sigma)
    estimator = QueryEstimator(
        cfg.query_estimator,
        query_fn=target_model.get_query_fn(),
        seed=cfg.experiment.seed + img_idx,
    )
    res_elpd = elpd_blend_attack(
        x_orig, label, surrogate, target_model,
        blender, estimator, cfg, run_logger=run_logger,
    )
    records.append({
        'img_idx': img_idx, 'label': label, 'method': 'elpd_blend',
        'success': res_elpd.success, 'queries': res_elpd.queries_used,
        'steps': res_elpd.steps_taken,
    })
    run_logger.end_image(res_elpd.success, res_elpd.queries_used)
    print(f'ELPD({'✓' if res_elpd.success else '✗'},{res_elpd.queries_used}q)  ', end='')

    # ── MI-FGSM ──────────────────────────────────────────────────────────
    if cfg.baselines.run_mifgsm:
        target_model.reset_count()
        res_mi = run_mifgsm(x_orig, label, surrogate, target_model, cfg)
        records.append({'img_idx': img_idx, 'label': label, 'method': 'mifgsm',
                        'success': res_mi.success, 'queries': 0, 'steps': res_mi.steps_taken})
        print(f'MI({'✓' if res_mi.success else '✗'})  ', end='')

    # ── DI-FGSM ──────────────────────────────────────────────────────────
    if cfg.baselines.run_difgsm:
        target_model.reset_count()
        res_di = run_difgsm(x_orig, label, surrogate, target_model, cfg)
        records.append({'img_idx': img_idx, 'label': label, 'method': 'difgsm',
                        'success': res_di.success, 'queries': 0, 'steps': res_di.steps_taken})
        print(f'DI({'✓' if res_di.success else '✗'})  ', end='')

    # ── NES-only ─────────────────────────────────────────────────────────
    if cfg.baselines.run_nes:
        target_model.reset_count()
        est_nes = QueryEstimator(
            cfg.query_estimator,
            query_fn=target_model.get_query_fn(),
            seed=cfg.experiment.seed + img_idx + 1000,
        )
        res_nes = run_nes_only(x_orig, label, target_model, est_nes, cfg)
        records.append({'img_idx': img_idx, 'label': label, 'method': 'nes_only',
                        'success': res_nes.success, 'queries': res_nes.queries_used,
                        'steps': res_nes.steps_taken})
        print(f'NES({'✓' if res_nes.success else '✗'},{res_nes.queries_used}q)  ', end='')

    # ── Static Hybrid ─────────────────────────────────────────────────────
    if cfg.baselines.run_static_hybrid.enabled:
        target_model.reset_count()
        est_sh = QueryEstimator(
            cfg.query_estimator,
            query_fn=target_model.get_query_fn(),
            seed=cfg.experiment.seed + img_idx + 2000,
        )
        res_sh = run_static_hybrid(x_orig, label, surrogate, target_model, est_sh, cfg)
        records.append({'img_idx': img_idx, 'label': label, 'method': 'static_hybrid',
                        'success': res_sh.success, 'queries': res_sh.queries_used,
                        'steps': res_sh.steps_taken})
        print(f'SH({'✓' if res_sh.success else '✗'},{res_sh.queries_used}q)  ', end='')

    # ── Square Attack ─────────────────────────────────────────────────────
    if cfg.baselines.run_square:
        target_model.reset_count()
        res_sq = run_square_attack(x_orig, label, target_model, cfg)
        records.append({'img_idx': img_idx, 'label': label, 'method': 'square',
                        'success': res_sq.success, 'queries': res_sq.queries_used,
                        'steps': res_sq.steps_taken})
        print(f'SQ({'✓' if res_sq.success else '✗'},{res_sq.queries_used}q)', end='')

    print()  # newline

run_logger.finish()
print(f'\nEvaluation complete: {len(records)} records.')

SyntaxError: invalid character '✓' (U+2713) (657893991.py, line 47)

In [7]:
# ── Cell 6: Results Table ─────────────────────────────────────────────────
df = pd.DataFrame(records)
df.to_csv('results/logs/final_results.csv', index=False)

summary = df.groupby('method').agg(
    ASR=('success', 'mean'),
    avg_queries=('queries', 'mean'),
    n_images=('img_idx', 'count'),
).round(4)
summary['ASR_pct'] = (summary['ASR'] * 100).round(2)
print(summary[['ASR_pct', 'avg_queries', 'n_images']].sort_values('ASR_pct', ascending=False))

NameError: name 'records' is not defined

In [ ]:
# ── Cell 7: Query vs ASR Curves ───────────────────────────────────────────
# Load the step-level CSV written by RunLogger for the ELPD attack,
# then reconstruct ASR-vs-queries curves for all methods.

QUERY_CHECKPOINTS = [100, 250, 500, 1000, 2000, 3000, 5000]

def asr_at_budget(df_method, budget):
    """Fraction of images where success happened within `budget` queries."""
    succeeded = df_method[df_method['success'] & (df_method['queries'] <= budget)]
    total     = len(df_method['img_idx'].unique())
    return len(succeeded['img_idx'].unique()) / max(total, 1)

methods        = df['method'].unique()
colours        = {
    'elpd_blend':   'steelblue',
    'mifgsm':       'green',
    'difgsm':       'limegreen',
    'nes_only':     'orange',
    'static_hybrid':'purple',
    'square':       'red',
}
linestyles     = {
    'elpd_blend':   '-',
    'mifgsm':       '--',
    'difgsm':       '--',
    'nes_only':     '-.',
    'static_hybrid':':',
    'square':       '-.',
}

fig, ax = plt.subplots(figsize=(10, 6))
for method in methods:
    df_m  = df[df['method'] == method]
    # For zero-query methods (pure transfer), ASR is constant
    if df_m['queries'].max() == 0:
        asr_val = df_m['success'].mean()
        ax.axhline(asr_val, color=colours.get(method, 'gray'),
                   linestyle=linestyles.get(method, '--'),
                   label=f'{method} (transfer-only, ASR={asr_val:.2%})', linewidth=2)
    else:
        qs   = QUERY_CHECKPOINTS
        asrs = [asr_at_budget(df_m, q) for q in qs]
        lw   = 2.5 if method == 'elpd_blend' else 1.5
        ax.plot(qs, asrs,
                color=colours.get(method, 'gray'),
                linestyle=linestyles.get(method, '-'),
                linewidth=lw,
                marker='o', markersize=4,
                label=method)

ax.axhline(0.9, color='black', linestyle=':', linewidth=1, alpha=0.5, label='90% ASR target')
ax.set_xlabel('Query Budget', fontsize=12)
ax.set_ylabel('Attack Success Rate (ASR)', fontsize=12)
ax.set_title('Query Efficiency: ELPD-Blend vs Baselines\n'
             f'Target={cfg.models.target.arch}, Surrogate={cfg.models.surrogate.arch}, ε={cfg.attack.epsilon}',
             fontsize=12)
ax.legend(fontsize=9, loc='lower right')
ax.grid(alpha=0.3)
ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig('results/plots/query_vs_asr.png', dpi=200)
plt.show()
print('Plot saved → results/plots/query_vs_asr.png')

In [ ]:
# ── Cell 8: η Trajectory Analysis ────────────────────────────────────────
# Load the step-level CSV from RunLogger and plot how η evolves over attack steps.
# A good attack should show η starting near 0.5, then adapting:
#   - rising toward 1.0 early in the attack (surrogate is useful near clean images)
#   - dropping toward 0.0 as the perturbation grows (surrogate becomes less reliable)

step_csv = 'results/logs/run_metrics.csv'
if os.path.exists(step_csv):
    df_steps = pd.read_csv(step_csv)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: mean η trajectory ± std band
    ax = axes[0]
    eta_by_step = df_steps.groupby('step')['eta_smoothed'].agg(['mean', 'std'])
    ax.plot(eta_by_step.index, eta_by_step['mean'], color='steelblue', linewidth=2, label='mean η')
    ax.fill_between(
        eta_by_step.index,
        (eta_by_step['mean'] - eta_by_step['std']).clip(0, 1),
        (eta_by_step['mean'] + eta_by_step['std']).clip(0, 1),
        alpha=0.25, color='steelblue', label='±1 std'
    )
    ax.set_xlabel('Attack Step')
    ax.set_ylabel('η (blending weight)')
    ax.set_title('η Trajectory (mean ± std across images)')
    ax.legend()
    ax.grid(alpha=0.3)

    # Right: cosine similarity between surrogate and target gradient over steps
    ax = axes[1]
    if 'cosine_sim_sur_tgt' in df_steps.columns:
        cos_by_step = df_steps.groupby('step')['cosine_sim_sur_tgt'].agg(['mean', 'std'])
        ax.plot(cos_by_step.index, cos_by_step['mean'], color='tomato', linewidth=2)
        ax.fill_between(
            cos_by_step.index,
            cos_by_step['mean'] - cos_by_step['std'],
            cos_by_step['mean'] + cos_by_step['std'],
            alpha=0.25, color='tomato'
        )
        ax.axhline(0, color='black', linestyle='--', linewidth=1)
        ax.set_xlabel('Attack Step')
        ax.set_ylabel('Cosine Similarity (surrogate vs target grad)')
        ax.set_title('Surrogate-Target Alignment over Steps')
        ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig('results/plots/eta_trajectory.png', dpi=200)
    plt.show()
    print('Plot saved → results/plots/eta_trajectory.png')
else:
    print('Step-level CSV not found — run Cell 5 first.')